# 02 Preprocess

In [1]:
# Load
import xarray as xr

STORE = "antarctic_oisst_2016_2020.zarr"  # cache path
ds = xr.open_zarr(STORE)  # fast local

ds

<xarray.Dataset> Size: 1GB
Dimensions:  (time: 1827, lat: 60, lon: 320)
Coordinates:
  * time     (time) datetime64[ns] 15kB 2016-01-01T12:00:00 ... 2020-12-31T12...
  * lat      (lat) float32 240B -74.88 -74.62 -74.38 ... -60.62 -60.38 -60.12
  * lon      (lon) float32 1kB 50.12 50.38 50.62 50.88 ... 129.4 129.6 129.9
Data variables:
    anom     (time, lat, lon) float64 281MB dask.array<chunksize=(31, 60, 320), meta=np.ndarray>
    err      (time, lat, lon) float64 281MB dask.array<chunksize=(31, 60, 320), meta=np.ndarray>
    ice      (time, lat, lon) float64 281MB dask.array<chunksize=(31, 60, 320), meta=np.ndarray>
    sst      (time, lat, lon) float64 281MB dask.array<chunksize=(31, 60, 320), meta=np.ndarray>
Attributes: (12/38)
    Conventions:                     CF-1.6, ACDD-1.3
    title:                           NOAA/NCEI 1/4 Degree Daily Optimum Inter...
    references:                      Reynolds, et al.(2007) Daily High-Resolu...
    source:                          ICOADS, NCEP_GTS, GSFC_ICE, NCEP_ICE, Pa...
    id:                              oisst-avhrr-v02r01.20201201.nc
    naming_authority:                gov.noaa.ncei
    ...                              ...
    time_coverage_end:               2020-12-01T23:59:59Z
    metadata_link:                   https://doi.org/10.25921/RE9P-PT57
    ncei_template_version:           NCEI_NetCDF_Grid_Template_v2.0
    comment:                         Data was converted from NetCDF-3 to NetC...
    sensor:                          Thermometer, AVHRR
    DODS_EXTRA.Unlimited_Dimension:  time

In [2]:
# standardise dataset

# Drop zlev 
if "zlev" in ds.dims:
  ds + ds.squeeze("zlev", drop=True)

# Ensure latitude is increasing 
if ds.lat[0] > ds.lat[-1]:
  ds = ds.sortby("lat")
  
# ENsure longitude is increasing
ds = ds.sortby("lon")

ds

<xarray.Dataset> Size: 1GB
Dimensions:  (time: 1827, lat: 60, lon: 320)
Coordinates:
  * time     (time) datetime64[ns] 15kB 2016-01-01T12:00:00 ... 2020-12-31T12...
  * lat      (lat) float32 240B -74.88 -74.62 -74.38 ... -60.62 -60.38 -60.12
  * lon      (lon) float32 1kB 50.12 50.38 50.62 50.88 ... 129.4 129.6 129.9
Data variables:
    anom     (time, lat, lon) float64 281MB dask.array<chunksize=(31, 60, 320), meta=np.ndarray>
    err      (time, lat, lon) float64 281MB dask.array<chunksize=(31, 60, 320), meta=np.ndarray>
    ice      (time, lat, lon) float64 281MB dask.array<chunksize=(31, 60, 320), meta=np.ndarray>
    sst      (time, lat, lon) float64 281MB dask.array<chunksize=(31, 60, 320), meta=np.ndarray>
Attributes: (12/38)
    Conventions:                     CF-1.6, ACDD-1.3
    title:                           NOAA/NCEI 1/4 Degree Daily Optimum Inter...
    references:                      Reynolds, et al.(2007) Daily High-Resolu...
    source:                          ICOADS, NCEP_GTS, GSFC_ICE, NCEP_ICE, Pa...
    id:                              oisst-avhrr-v02r01.20201201.nc
    naming_authority:                gov.noaa.ncei
    ...                              ...
    time_coverage_end:               2020-12-01T23:59:59Z
    metadata_link:                   https://doi.org/10.25921/RE9P-PT57
    ncei_template_version:           NCEI_NetCDF_Grid_Template_v2.0
    comment:                         Data was converted from NetCDF-3 to NetC...
    sensor:                          Thermometer, AVHRR
    DODS_EXTRA.Unlimited_Dimension:  time

In [4]:
# Ice mask

ICE_THRESHOLD = 15.0

sst = ds["sst"].where(ds['ice'] < ICE_THRESHOLD)

# Sanity cehck on missingness
nan_frac = float(sst.isnull().mean())
print("NaN fraction after ice mask: ", nan_frac)


NaN fraction after ice mask:  0.721411010764459


In [ ]:
# Daily climateology
# day of year index
day = sst["time"].dt.dayofyear

# daily climatology: dims -> (dayofyear, at, lon)
clim = sst.groupby(day).mean("time", skipna=True)

print(clim)

In [ ]:
# COmpute Anomalies
anom = sst.groupby(day) - clim

print(anom)


In [ ]:
# plot

import matplotlib.pyplot as plt

t0 = anom.time.values[0]
plt.figure(figsize(15,6))
anom.sel(time=t0).plot()
plt.title(f"SST anomaly (ice-masked) - {str(t0)[:10]}")
plt.show()

# Phsyics-informed residuals
Implementing local spacial expectation of anomaly:
$$
\hat a(x,t)=\sum_{x’\in \mathcal{N}(x)} w(x,x’)\,a(x’,t)
$$
with weights based on great-circle distance d:
$$
w \propto \exp\!\left(-\frac{d^2}{2\ell^2}\right)
$$

In [ ]:
# Physics informed residuals
from sklearn.neighbours import BallTree
from scipy import sparse

EARTH_R_KM = 6371.0

def build_knn_weight_matrix(
  anom: xr.DataArray,
  k: int = 25,
  radius_km: float = 300.0,
  ell_km: float = 150.0, 
):
  """
    Build sparse weight matrix W over valid (non-NaN) grid points using haversine kNN.
    Weights are Gaussian in great-circle distance and truncated at radius_km.
    """
  
  # Stack spacial dims -> points
  A = anom.stack(points=("lat", "lon"))
  
  # Define "valid points" based on time=0 snapshot
  valid = ~np.isnan(A.isel(time=0).values)
  valid_idx = np.where(valid)[0]
  if valid_idx.size == 0:
    raise ValueError("No valid points found. CHeck ice mask / region selection.")
  
  # Coordinates for valid points
  lat = A["lat"].values
  lon = A["lon"].values
  pts_lat = np.repeat(lat, len(lon))    # (lat, lon) grid flattened
  pts_lon = np.tile(lon, len(lat))
  
  vlat = ptslat[valid_idx]
  vlon = ptslon[valid_idx]
  
  # BallTree expects radians in [lat, lon]
  X = np.deg2rad(np.c_[vlat, vlon])
  tree = BallTree(X, metric="haversine")
  
  # Query k+1 so we can drop self-neighbour at distance 0
  dist_rad, ind = tree.query(X, k=k+1)
  dist_km = dist_rad * EARTH_R_KM
  
  # Drop self neighbour in column 0
  dist_km = dist_km[:, 1:]
  ind = ind[:, 1:]            # indices in valid-point space [0..Nv-1]
  
  # Gaussian weights with lengthscale ell_km, then truncate beyond radius_km
  # w = exp( - d^2 / (2 ell^2) )
  w = np.exp(-(dist_km**2) / (2.0 * ell_km**2))
  w[dist_km > radius_km] = 0.0
  
  # Row-normalise (avoid divide by zero)
  row_sum = w.sum(axis=1, keepdims=True)
  row_sum[row_sum == 0.0] = np.nan
  w = w / row_sum

  # Build sparse matrix W (Nv x Nv) in CSR format
  Nv = valid_idx.size
  rows = np.repeat(np.arange(Nv), k)
  cols = ind.reshape(-1)
  data = w.reshape(-1)

  # Remove NaNs and zeros
  ok = np.isfinite(data) & (data != 0.0)
  W = sparse.csr_matrix((data[ok], (rows[ok], cols[ok])), shape=(Nv, Nv))

  # Return mapping objects to reconstruct full grid
  template = A.isel(time=0).copy(deep=False)  # shape (points,)
  return W, valid_idx, template

    
  
  
  

In [ ]:
def expected_anomaly_sparse(anom: xr.DataArray, W, valid_idx, template_points):
    """
    Compute expected anomaly using sparse W with missing-aware normalization.
    Returns DataArray with same dims as anom.
    """
    A = anom.stack(points=("lat", "lon"))

    def _apply_W(block):
        # block shape: (time, points) as numpy
        # we'll operate only on valid points
        out = np.full_like(block, np.nan, dtype=np.float32)

        V = block[:, valid_idx].astype(np.float32)  # (T, Nv)
        M = np.isfinite(V).astype(np.float32)
        V0 = np.nan_to_num(V, nan=0.0)

        # For each time: expected = (W @ V0) / (W @ M)
        # We'll do this in batch using matrix multiply:
        # (Nv x Nv) @ (Nv x T) -> (Nv x T), then transpose
        num = (W @ V0.T).T  # (T, Nv)
        den = (W @ M.T).T   # (T, Nv)

        with np.errstate(invalid="ignore", divide="ignore"):
            E = num / den
        E[den == 0.0] = np.nan

        out[:, valid_idx] = E
        return out

    # Apply over dask chunks (time chunks) safely
    E_points = xr.apply_ufunc(
        _apply_W,
        A,
        input_core_dims=[["points"]],
        output_core_dims=[["points"]],
        dask="parallelized",
        output_dtypes=[np.float32],
    )

    # Unstack back to (time, lat, lon)
    E = E_points.unstack("points").transpose("time", "lat", "lon")
    return E

In [ ]:
# anom: (time, lat, lon) and already ice-masked
W, valid_idx, template = build_knn_weight_matrix(
    anom,
    k=25,
    radius_km=300.0,
    ell_km=150.0
)

expected = expected_anomaly_sparse(anom, W, valid_idx, template)
resid = anom - expected

In [ ]:
import matplotlib.pyplot as plt

t0 = resid.time.values[0]
plt.figure(figsize=(8,4))
resid.sel(time=t0).plot()
plt.title(f"Residual anomaly — {str(t0)[:10]}")
plt.show()